# **LangChain Memory**

### **Objective:**
Understand how conversational memory works in LangChain and compare four strategies for deciding what the model gets to "remember": keep everything, keep the last few turns, keep a token budget, or summarize old turns.

---

### **Core concept:**
An LLM is **stateless**. Every call only knows what is in the messages you send. "Memory" is just code that decides **which past messages are re-sent** with each new question. The four strategies differ only in that decision.

---

### **Note:**
- Set `OPENAI_API_KEY` in your environment.
- Tested against `langchain 1.3.x`, `langchain-core 1.7.x`, `langchain-openai 1.2.x`. In LangChain 1.x the old `ConversationBufferMemory`-style classes are no longer in `langchain.memory`, and `RunnableWithMessageHistory` is deprecated. This notebook therefore implements each strategy in a few lines on top of `BaseChatMessageHistory` so you can see exactly how it works, and passes the history to the prompt directly.

---

### **Steps to perform:**
1. Import the necessary modules and helpers
2. Initialize the chat model and a `chat()` helper
3. See why memory is needed (stateless model)
4. Buffer memory: keep everything (`ConversationBufferMemory`)
5. Window memory: keep the last k turns (`ConversationBufferWindowMemory`, k = 1)
6. Token buffer memory: keep at most 30 tokens (`ConversationTokenBufferMemory`)
7. Summary buffer memory: summarize overflow beyond 100 tokens (`ConversationSummaryBufferMemory`)

---

**Note:** Follow Demo 01: Langchain Model Output Parser Step 1 for setting up the Environment **(Not required if it is already set up)**

### **Step 1: Import the necessary modules and helpers**
- `BaseChatMessageHistory` is the interface all LangChain message stores implement (`messages`, `add_messages`, `clear`)
- `count_tokens` counts tokens with `tiktoken` (the tokenizer used by recent OpenAI models). If the tokenizer file cannot be downloaded, it falls back to a rough 4-characters-per-token estimate

In [ ]:
import os
import warnings

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

warnings.filterwarnings("ignore")

try:
    import tiktoken
    _enc = tiktoken.get_encoding("o200k_base")

    def count_tokens(text: str) -> int:
        return len(_enc.encode(text))
except Exception:  # offline / blocked download
    def count_tokens(text: str) -> int:
        return max(1, len(text) // 4)

print("Tokens in 'Machine Learning is what?':", count_tokens("Machine Learning is what?"))

- Helper to print a memory's contents with the token count of each message

In [ ]:
def print_clean_history(memory):
    total = 0
    for m in memory.messages:
        role = {"human": "User", "ai": "AI", "system": "System"}.get(m.type, m.type)
        n = count_tokens(m.content)
        total += n
        print(f"[{n:>3} tok] {role}: {m.content}")
    print(f"Total: {len(memory.messages)} messages, {total} tokens\n")

### **Step 2: Initialize the chat model and a `chat()` helper**
- `gpt-5-mini` is a reasoning model and only supports its default sampling settings, so we do not set `temperature`; outputs vary slightly between runs
- `chat()` does what every memory strategy needs: (1) read the stored history, (2) send history + new question to the model, (3) save the new question and answer back into the memory

In [ ]:
assert os.getenv("OPENAI_API_KEY"), "Set the OPENAI_API_KEY environment variable first."

llm = ChatOpenAI(model="gpt-5-mini")

In [ ]:
def chat(memory, user_input, system="You are a helpful assistant. Reply in one short sentence."):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("placeholder", "{history}"),   # the stored messages are inserted here
        ("human", "{input}"),
    ])
    sent = list(memory.messages)        # exactly what the model will see as history
    reply = (prompt | llm).invoke({"history": sent, "input": user_input}).content
    memory.add_messages([HumanMessage(content=user_input), AIMessage(content=reply)])
    print(f"User: {user_input}\nAI:   {reply}\n      (model was sent {len(sent)} history messages)\n")
    return reply

### **Step 3: See why memory is needed**
Two separate calls with no history. The second call has no idea about the first.

In [ ]:
response1 = llm.invoke("My name is Alex.")
print("response1", response1.content)

response2 = llm.invoke("What is my name?")
print("response2", response2.content)

Without history the model cannot answer. Every strategy below fixes this by re-sending past messages, and they differ in **how much** they re-send.

### **Step 4: Buffer memory (`ConversationBufferMemory`)**
- Stores the entire conversation and re-sends all of it on every call

In [ ]:
class BufferMemory(BaseChatMessageHistory):
    """Keeps every message. Subclasses override _trim() to forget some."""

    def __init__(self):
        self._messages = []

    @property
    def messages(self):
        return self._messages

    def add_messages(self, messages):
        self._messages.extend(messages)
        self._trim()

    def _trim(self):
        pass  # buffer memory never forgets

    def clear(self):
        self._messages = []

In [ ]:
CONVERSATION = [
    "Hello, my name is Alex",
    "What is 2+2?",
    "Add 10 to that result",
    "What is my name?",
]

print("### Buffer memory ###\n")
buffer_memory = BufferMemory()
for q in CONVERSATION:
    chat(buffer_memory, q)

print("Memory contents:")
print_clean_history(buffer_memory)

**What to observe**
- "Add 10 to that result" works (the model sees the earlier `2+2` answer) and "What is my name?" works (it sees the first message).
- The history sent to the model grows every turn (0, 2, 4, 6 messages), so cost and latency grow with conversation length and will eventually hit the context limit.
- Best for short conversations where nothing can be lost.

### **Step 5: Window memory (`ConversationBufferWindowMemory`, k = 1)**
- Keeps only the last `k` turns (one turn = one user message + one AI message)

In [ ]:
class WindowMemory(BufferMemory):
    def __init__(self, k=1):
        super().__init__()
        self.k = k

    def _trim(self):
        self._messages = self._messages[-2 * self.k:]

In [ ]:
print("### Window memory (k=1) ###\n")
window_memory = WindowMemory(k=1)
for q in CONVERSATION:
    chat(window_memory, q)

print("Memory contents:")
print_clean_history(window_memory)

**What to observe**
- "Add 10 to that result" still works because the previous turn (`2+2`) is inside the window.
- "What is my name?" fails: the first turn fell out of the window, so the model no longer knows Alex.
- The history sent to the model never exceeds 2 messages, so cost stays flat. The price is that anything older than `k` turns is forgotten.

### **Step 6: Token buffer memory (`ConversationTokenBufferMemory`, 30 tokens)**
- Keeps the most recent messages whose total size is at most `max_tokens`. When the limit is exceeded, the **oldest** messages are dropped first
- Size is measured in real tokens (`count_tokens`), not words
- Replies are limited to a few words via the system prompt, otherwise a single long answer (more than 30 tokens) would not fit in the budget at all and would be dropped immediately

In [ ]:
class TokenBufferMemory(BufferMemory):
    def __init__(self, max_tokens=30):
        super().__init__()
        self.max_tokens = max_tokens

    def token_count(self):
        return sum(count_tokens(m.content) for m in self._messages)

    def _trim(self):
        while self._messages and self.token_count() > self.max_tokens:
            self._messages.pop(0)

In [ ]:
print("### Token buffer memory (max 30 tokens) ###\n")
token_memory = TokenBufferMemory(max_tokens=30)
SHORT = "You are a helpful assistant. Answer in at most 8 words."

for q in [
    "Machine Learning is what?",
    "Neural Networks are what?",
    "AI Assistants are what?",
    "Which topic did I ask about first?",
]:
    chat(token_memory, q, system=SHORT)
    print(f"      tokens in memory: {token_memory.token_count()} / {token_memory.max_tokens}\n")
    assert token_memory.token_count() <= token_memory.max_tokens

print("Memory contents:")
print_clean_history(token_memory)

**What to observe**
- After every turn the token count printed is never above 30. Old messages are removed one by one from the front until the total fits.
- Unlike the window, the amount kept depends on message **size**: short messages mean more turns are kept, long messages mean fewer.
- The first question (Machine Learning) is eventually dropped, so the last question cannot be answered.
- Token counts include message text only. The API adds a few tokens of formatting per message, so use a safety margin in production.
- Use this when you must stay inside a fixed token budget.

### **Step 7: Summary buffer memory (`ConversationSummaryBufferMemory`, 100 tokens)**
- Recent messages are kept word for word, up to `max_tokens`
- Messages that overflow the limit are not discarded: the LLM folds them into a running summary, which is sent to the model as a system message ahead of the recent messages
- Each overflow costs one extra LLM call

In [ ]:
SUMMARY_PROMPT = ChatPromptTemplate.from_template(
    "Progressively summarize the conversation. Keep names, facts and decisions.\n\n"
    "Current summary:\n{summary}\n\n"
    "New lines to add:\n{new_lines}\n\n"
    "New summary:"
)


class SummaryBufferMemory(BufferMemory):
    def __init__(self, llm, max_tokens=100):
        super().__init__()
        self.llm = llm
        self.max_tokens = max_tokens
        self.summary = ""

    @property
    def messages(self):
        if self.summary:
            return [SystemMessage(content=f"Summary of earlier conversation: {self.summary}")] + self._messages
        return self._messages

    def buffer_tokens(self):
        return sum(count_tokens(m.content) for m in self._messages)

    def _trim(self):
        pruned = []
        while self._messages and self.buffer_tokens() > self.max_tokens:
            pruned.append(self._messages.pop(0))
        if pruned:
            new_lines = "\n".join(f"{m.type}: {m.content}" for m in pruned)
            self.summary = (SUMMARY_PROMPT | self.llm).invoke(
                {"summary": self.summary or "(none)", "new_lines": new_lines}
            ).content

    def clear(self):
        super().clear()
        self.summary = ""

In [ ]:
print("### Summary buffer memory (max 100 tokens) ###\n")
summary_memory = SummaryBufferMemory(llm, max_tokens=100)
BRIEF = "You are a helpful assistant. Answer in at most 25 words."

for q in [
    "Hi, I'm Alex and I'm learning LangChain.",
    "I code in Python and use Pinecone for vector search.",
    "I'm building a RAG chatbot over PDF documents.",
    "My project deadline is next Friday.",
    "Summarize what you know about me and my project.",
]:
    chat(summary_memory, q, system=BRIEF)
    print(f"      buffer tokens: {summary_memory.buffer_tokens()} / {summary_memory.max_tokens}")
    print(f"      summary: {summary_memory.summary or '(none yet)'}\n")

print("Memory contents (what the model would be sent next):")
print_clean_history(summary_memory)

**What to observe**
- The first turns are kept verbatim. Once the buffer exceeds 100 tokens, the oldest messages disappear from the buffer and appear in the `summary` instead.
- The final answer still mentions Alex, Python, Pinecone, RAG and the deadline even though those early messages are no longer in the buffer: the summary carried them.
- Trade-off: details can be lost or distorted in the summary, and each summarization costs an extra LLM call.
- Best for long sessions (customer support, personal assistants) where the gist matters more than exact wording.

### **Conclusion**
All four strategies implement the same interface and differ only in what they keep:

| Strategy | LangChain legacy name | Keeps | Cost per call | Loses |
|---|---|---|---|---|
| Buffer | `ConversationBufferMemory` | Everything | Grows without limit | Nothing |
| Window | `ConversationBufferWindowMemory` | Last k turns | Constant | Anything older than k turns |
| Token buffer | `ConversationTokenBufferMemory` | Most recent messages within a token budget | Bounded | Oldest messages |
| Summary buffer | `ConversationSummaryBufferMemory` | Recent messages + summary of older ones | Bounded, plus summarization calls | Fine detail of old turns |

Choose by what you can afford to forget and how much context you can afford to send.

For production on LangChain 1.x, persist conversation state with LangGraph checkpointers (`langgraph.checkpoint`) and trim or summarize with `langchain_core.messages.trim_messages`. The concepts above are the same.

---